In [47]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential 
from tensorflow.keras.layers import Dense,Dropout,Convolution2D,MaxPooling2D,Flatten
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import keras_tuner as kt

In [48]:
# import os
# import shutil
# import random

# # source folders
# source_cat = "/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dog_vs_cat_microsoft/PetImages/Cat"
# source_dog = "/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dog_vs_cat_microsoft/PetImages/Dog"

# # destination base
# base_dir = "/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dataset"

# # create folders
# for split in ['train', 'test']:
#     for cls in ['cats', 'dogs']:
#         os.makedirs(os.path.join(base_dir, split, cls), exist_ok=True)

# # function to split
# def split_data(source, train_dir, test_dir, split_ratio=0.8):
#     files = [f for f in os.listdir(source) if os.path.isfile(os.path.join(source, f))]
#     random.shuffle(files)

#     split_index = int(len(files) * split_ratio)
#     train_files = files[:split_index]
#     test_files = files[split_index:]

#     for f in train_files:
#         shutil.copy(os.path.join(source, f), os.path.join(train_dir, f))

#     for f in test_files:
#         shutil.copy(os.path.join(source, f), os.path.join(test_dir, f))

# # split cats
# split_data(
#     source_cat,
#     os.path.join(base_dir, 'train/cats'),
#     os.path.join(base_dir, 'test/cats')
# )

# # split dogs
# split_data(
#     source_dog,
#     os.path.join(base_dir, 'train/dogs'),
#     os.path.join(base_dir, 'test/dogs')
# )

# print("Dataset structured successfully!")

In [49]:
import os
import random

def delete_random_files(dir_path, n):
    # get all files (ignore folders)
    files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]
    
    if n > len(files):
        print(f"Only {len(files)} files available. Deleting all.")
        n = len(files)
    
    # pick random files
    to_delete = random.sample(files, n)
    
    # delete them
    for file in to_delete:
        file_path = os.path.join(dir_path, file)
        os.remove(file_path)
    
    print(f"Deleted {len(to_delete)} files from {dir_path}")

In [50]:
delete_random_files('/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/train/cats',12000)
delete_random_files('/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/train/dogs',12000)

Deleted 12000 files from /Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/train/cats
Deleted 12000 files from /Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/train/dogs


In [51]:
keras.utils.image_dataset_from_directory(
    directory='/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/train',
    labels="inferred",
    label_mode="int",
    class_names=None,
    color_mode="rgb",
    batch_size=32,
    image_size=(256, 256),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
    pad_to_aspect_ratio=False,
    data_format=None,
    format="tf",
    verbose=True,
)
keras.utils.image_dataset_from_directory(
    directory='/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/validation',
    labels="inferred",
    label_mode="int",
    class_names=None,
    color_mode="rgb",
    batch_size=32,
    image_size=(256, 256),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
    pad_to_aspect_ratio=False,
    data_format=None,
    format="tf",
    verbose=True,
)



Found 1000 files belonging to 2 classes.
Found 8000 files belonging to 2 classes.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [52]:

def process(image,label):
    return image / 255.,label

In [53]:
train=tensorflow.keras.utils.image_dataset_from_directory('/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/train').map(process)
validate=tensorflow.keras.utils.image_dataset_from_directory('/Users/adarsh/PycharmProjects/Deep Learning/Neural_Networks/CNN/dogcat/validation').map(process)

Found 1000 files belonging to 2 classes.
Found 8000 files belonging to 2 classes.


In [54]:
train

<_MapDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [55]:
model=Sequential()

model.add(Convolution2D(32,kernel_size=(3,3),padding='valid',activation='relu',input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Convolution2D(32,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Convolution2D(32,kernel_size=(3,3),padding='valid',activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(64,activation='relu'))
model.add(Dense(32,activation='relu'))
model.add(Dense(1,activation='sigmoid'))


In [56]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 125, 125, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 60, 60, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │     3,686,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,716,289 (14.18 MB)

 Trainable params: 3,716,289 (14.18 MB)

 Non-trainable params: 0 (0.00 B)

In [57]:
model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

In [58]:
import os
import random

def delete_random_files(dir_path, n):
    # get all files (ignore folders)
    files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]
    
    if n > len(files):
        print(f"Only {len(files)} files available. Deleting all.")
        n = len(files)
    
    # pick random files
    to_delete = random.sample(files, n)
    
    # delete them
    for file in to_delete:
        file_path = os.path.join(dir_path, file)
        os.remove(file_path)
    
    print(f"Deleted {len(to_delete)} files from {dir_path}")

In [59]:
def length_file(dir_path):
    files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))]
    return len(files)

In [60]:
model.fit(train,epochs=10,validation_data=validate,batch_size=50,verbose=True)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4820 - loss: 0.7092

KeyboardInterrupt: 